<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 강화학습·에이전트 · 02. Double DQN

## Deep Reinforcement Learning with Double Q-learning

- **원 논문:** [Deep Reinforcement Learning with Double Q-learning](https://ojs.aaai.org/index.php/AAAI/article/view/10295)
- **저자 / 발표:** Hado van Hasselt, Arthur Guez, and David Silver · AAAI (2016)
- **난이도 / 예상 시간:** 중급 · 약 55분
- **선수 지식:** DQN, maximization bias, independent estimators
- **로컬 학습 데이터:** `data/field_curriculum/gridworld.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** online action selection과 target evaluation을 분리하고 noisy-value overestimation을 계량한다.

**축소하거나 생략한 부분:** 논문은 Atari 49 games에서 DQN과 Double DQN을 비교한다. 축소판은 논문의 noisy action-value 현상과 Double target을 로컬 MDP에서 재현한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2, Eq. (3) | DQN max target | 같은 estimator가 선택과 평가를 모두 하는 target을 비교 기준으로 둔다. |
| §2, Eq. (4); §3 Double DQN displayed target | decoupled selection/evaluation | argmax online, evaluate target 순서를 구현한다. |
| §2, Theorem 1 and Figure 1 | overestimation experiment | zero-valued noisy actions에서 max estimator의 positive bias를 측정한다. |

## 핵심 재현

        Double DQN target은 $a^*=\arg\max_a Q_{online}(s',a)$로 고르고
        $Q_{target}(s',a^*)$로 평가합니다. 두 noise estimator를 분리해 max bias도 직접 측정합니다.

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(421)
np.random.seed(421)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# Tiny scalar/table loops may be faster on CPU even when a GPU exists;
# AI_LAB_DEVICE=cpu remains the low-overhead override.
DATA_PATH = Path("data/field_curriculum/gridworld.json")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
config = json.loads(DATA_PATH.read_text(encoding="utf-8"))
required = {
    "seed",
    "size",
    "start_state",
    "terminal_state",
    "gamma",
    "transitions",
    "bandit_means",
    "offline_action_noise",
}
assert required.issubset(config), f"JSON 키 불일치: {sorted(config)}"
rows = config["transitions"]
states = torch.tensor([int(r["state"]) for r in rows], dtype=torch.long)
actions = torch.tensor([int(r["action"]) for r in rows], dtype=torch.long)
rewards = torch.tensor([float(r["reward"]) for r in rows], dtype=torch.float32)
next_states = torch.tensor([int(r["next_state"]) for r in rows], dtype=torch.long)
dones = torch.tensor([float(r["done"]) for r in rows], dtype=torch.float32)
n_states = int(max(states.max(), next_states.max()).item()) + 1
n_actions = int(actions.max().item()) + 1
gamma = float(config["gamma"])
start_state, terminal_state = int(config["start_state"]), int(config["terminal_state"])
transition_table = {
    (int(r["state"]), int(r["action"])): (
        int(r["next_state"]),
        float(r["reward"]),
        bool(r["done"]),
    )
    for r in rows
}


def encode(s):
    return F.one_hot(torch.as_tensor(s, dtype=torch.long), n_states).float()


def encode_device(s):
    return encode(s).to(DEVICE)


def env_step(state, action):
    return transition_table[(int(state), int(action))]


def value_iteration(iterations=300):
    q = torch.zeros(n_states, n_actions)
    for _ in range(iterations):
        old = q.clone()
        for (s, a), (ns, r, done) in transition_table.items():
            q[s, a] = r if done else r + gamma * old[ns].max()
    return q


q_star = value_iteration()
assert torch.isfinite(q_star).all() and len(rows) > 0
print(ACCELERATOR.summary())
print(
    "RL environment, replay metadata, and tree search stay on CPU; "
    "neural batches use DEVICE."
)
print(
    f"local MDP: states={n_states}, actions={n_actions}, "
    f"transitions={len(rows)}, gamma={gamma}"
)

## 포트폴리오 구현 설계: 수식 → 에이전트 책임 → 검증

논문의 핵심 update를 실행 가능한 에이전트의 `forward`, `loss`, `update`,
`evaluate`로 나눕니다. 아래 식의 선택·평가·gradient 경계를 메서드 본문에서
한 줄씩 추적하세요.

$$
a^*=\arg\max_aQ_\theta(s',a),\qquad y=r+\gamma(1-d)Q_{\theta^-}(s',a^*)
$$

- **기호와 역할:** $Q_\theta$는 action을 선택하고 $Q_{\theta^-}$는 그 action을 평가해 maximization bias를 분리합니다.
- **shape/state 계약:** `online_next/target_next [B, A] → chosen [B, 1] → target [B]`
- **논문 위치:** `§2, Eq. (3)`의 **DQN max target**
- **코드 Task:** 선택/평가 분리 target, noisy max bias 실험과 Double TD update를 구현합니다.
- **학습·평가 흐름:** online argmax → target gather → detached TD loss → sync → bias 비교
- **원문 대비 한계:** 논문은 Atari 49 games에서 DQN과 Double DQN을 비교한다. 축소판은 논문의 noisy action-value 현상과 Double target을 로컬 MDP에서 재현한다.
- **구현 이유:** target, gradient, optimizer 책임을 Agent 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 Bellman target, policy objective, search score 같은 핵심 계산을
불투명한 trainer 뒤에 감추지 않습니다. 실습본에서는 공개 API를 유지하면서 TODO를
채우고, 각 메서드의 입력 shape와 gradient가 끊기는 위치를 설명하세요.

In [ ]:
class DoubleDqnAgent(nn.Module):
    """Double DQN의 핵심 학습·평가 경로. 입출력 계약: online_next/target_next [B, A] → chosen [B,
    1] → target [B]."""

    def __init__(self, online_network, target_network, optimizer, discount):
        """network, optimizer, target 또는 table 상태를 명시적으로 저장한다."""
        super().__init__()
        self.online_network = online_network
        self.target_network = target_network
        self.optimizer = optimizer
        self.discount = discount

    def config(self):
        """재현 범위와 핵심 하이퍼파라미터를 dict로 반환한다."""
        return {"selection": "online", "evaluation": "target", "gamma": self.discount}

    def forward(self, state):
        """policy/value/Q 순전파를 계산한다. shape 계약: online_next/target_next [B, A] → chosen
        [B, 1] → target [B]."""
        return self.online_network(encode_device(state))

    def loss(self, state, action, reward, next_state, done):
        """논문의 목적 함수를 미분 가능한 scalar tensor로 반환한다."""
        predicted_q = self.forward(state).gather(1, action[:, None]).squeeze(1)
        with torch.no_grad():
            online_next_q = self.online_network(encode_device(next_state))
            selected_action = online_next_q.argmax(dim=1, keepdim=True)
            target_next_q = self.target_network(encode_device(next_state))
            evaluated_q = target_next_q.gather(1, selected_action).squeeze(1)
            target_q = reward + self.discount * (1.0 - done) * evaluated_q
        return F.mse_loss(predicted_q, target_q)

    def update(self, state, action, reward, next_state, done):
        """zero_grad, backward, step을 수행하고 유한한 float loss를 반환한다."""
        objective = self.loss(state, action, reward, next_state, done)
        self.optimizer.zero_grad()
        objective.backward()
        self.optimizer.step()
        return float(objective.detach())

    def sync_target(self):
        """online network의 가중치를 target network로 복사한다."""
        self.target_network.load_state_dict(self.online_network.state_dict())

    def evaluate(self, state):
        """no_grad 경로에서 정책 또는 value 품질 지표 dict를 반환한다."""
        with torch.no_grad():
            online_q = self.forward(state)
            selected_action = online_q.argmax(dim=1, keepdim=True)
            target_q = self.target_network(encode_device(state))
            evaluated_q = target_q.gather(1, selected_action).squeeze(1)
        return {
            "selected_actions": selected_action.squeeze(1).tolist(),
            "evaluated_mean": float(evaluated_q.mean()),
        }

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `§2, Eq. (4); §3 Double DQN displayed target` — **decoupled selection/evaluation**
- **구현 이유:** argmax online, evaluate target 순서를 구현한다.
- **읽을 코드:** 아래 `implementation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
def double_target(r, done, online_next, target_next, discount):
    chosen = online_next.argmax(1, keepdim=True)
    evaluated = target_next.gather(1, chosen).squeeze(1)
    return (r + discount * (1 - done) * evaluated).detach()


y = double_target(
    torch.tensor([1.0]),
    torch.tensor([0.0]),
    torch.tensor([[1.0, 4.0]]),
    torch.tensor([[8.0, 2.0]]),
    0.5,
)
assert torch.allclose(y, torch.tensor([2.0]))

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `§2, Theorem 1 and Figure 1` — **overestimation experiment**
- **구현 이유:** zero-valued noisy actions에서 max estimator의 positive bias를 측정한다.
- **읽을 코드:** 아래 `bias-experiment` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
rng = np.random.default_rng(422)
online_noise = rng.normal(size=(5000, 8))
target_noise = rng.normal(size=(5000, 8))
dqn_est = online_noise.max(1)
chosen = online_noise.argmax(1)
double_est = target_noise[np.arange(len(chosen)), chosen]
dqn_bias = float(dqn_est.mean())
double_bias = float(double_est.mean())
print(f"DQN bias={dqn_bias:.3f}, Double bias={double_bias:.3f}")
assert dqn_bias > 0.8 and abs(double_bias) < 0.08

### 단계 4. 평가·시각화

- **논문의 어느 부분인가:** `§2, Eq. (3)` — **DQN max target**
- **구현 이유:** 같은 estimator가 선택과 평가를 모두 하는 target을 비교 기준으로 둔다.
- **읽을 코드:** 아래 `training-evaluation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
online = nn.Sequential(nn.Linear(n_states, 24), nn.ReLU(), nn.Linear(24, n_actions)).to(
    DEVICE
)
target = nn.Sequential(nn.Linear(n_states, 24), nn.ReLU(), nn.Linear(24, n_actions)).to(
    DEVICE
)
target.load_state_dict(online.state_dict())
opt = torch.optim.Adam(online.parameters(), lr=0.012)
gen = torch.Generator().manual_seed(423)
history = []

portfolio_agent = DoubleDqnAgent(online, target, opt, gamma)
initial_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in online.parameters()]
)
for step in range(220):
    index = torch.randint(len(rows), (min(32, len(rows)),), generator=gen)
    state, action, reward, next_state, done = ACCELERATOR.move(
        states[index],
        actions[index],
        rewards[index],
        next_states[index],
        dones[index],
    )
    history.append(portfolio_agent.update(state, action, reward, next_state, done))
    if (step + 1) % 20 == 0:
        portfolio_agent.sync_target()
final_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in online.parameters()]
)
parameter_delta = float((final_parameters - initial_parameters).abs().sum())
portfolio_metrics = portfolio_agent.evaluate(torch.arange(n_states))
assert parameter_delta > 0.0 and np.isfinite(history).all()
assert len(portfolio_metrics["selected_actions"]) == n_states

선택과 평가 noise가 완전히 상관되면 다시 max estimator와 같아져 overestimation 완화가 사라집니다. 지연 target이 두 estimator를 부분적으로 분리합니다.

### 단계 5. 통합 Agent가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `§2, Eq. (3)`의 **DQN max target**
- **핵심 식:**

  $$
  a^*=\arg\max_aQ_\theta(s',a),\qquad y=r+\gamma(1-d)Q_{\theta^-}(s',a^*)
  $$

- **입출력 shape:** `online_next/target_next [B, A] → chosen [B, 1] → target [B]`
- **구현 이유:** 앞 셀은 target, objective, search를 작은 연산으로
  분해해 확인하고, 이 셀은 같은 구성 요소를
  `DoubleDqnAgent`에 주입해 최종 학습·평가 경로를
  하나로 묶습니다. 같은 estimator가 선택과 평가를 모두 하는 target을 비교 기준으로 둔다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 정책·가치 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

In [ ]:
config_contract = portfolio_agent.config()
numeric_metrics = [
    float(value)
    for value in portfolio_metrics.values()
    if isinstance(value, (int, float))
]
assert config_contract and len(history) > 0
assert np.isfinite(np.asarray(history, dtype=float)).all()
assert parameter_delta > 0.0
assert numeric_metrics and np.isfinite(numeric_metrics).all()
print(
    {
        "class": type(portfolio_agent).__name__,
        "steps": len(history),
        "parameter_delta": parameter_delta,
        "metrics": portfolio_metrics,
    }
)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 논문은 Atari 49 games에서 DQN과 Double DQN을 비교한다. 축소판은 논문의 noisy action-value 현상과 Double target을 로컬 MDP에서 재현한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.